# Reproducibility Demonstration

Demonstrates reproducing a tracking-trajectory result directly from the
codebase (no stored per-timestep trace files are shipped -- trajectories
are always regenerated live from `src/scenarios.py`, the same code path
used to produce the paper's trajectory figures).

**Steps:**
1. Build the validated PV model and the 5 MPPT algorithms
2. Re-simulate the steady-state (STC) scenario for P&O
3. Recreate a tracking-trajectory plot
4. Recompute tracking efficiency and compare against `results/summary_table.csv`

In [ ]:
import sys
sys.path.append('..')
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('../styles/publication.mplstyle')
%matplotlib inline

In [ ]:
from src import config
from src.pv_model import TwoDiodeModel, extract_two_diode_parameters
from src.scenarios import build_default_algorithms, scenario_steady_state, run_scenario

panel_params = extract_two_diode_parameters(
    voc=config.PANEL_VOC_STC, isc=config.PANEL_ISC_STC,
    vmp=config.PANEL_VMP_STC, imp=config.PANEL_IMP_STC,
)
pv_model = TwoDiodeModel(panel_params, num_cells=config.PANEL_NS)
algorithms = build_default_algorithms(pv_model)
result = run_scenario(scenario_steady_state(), algorithms['p_and_o'], pv_model)
print(f"Simulated {len(result.times)} time steps for P&O at STC")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(result.times, result.powers, label='Output Power', linewidth=2)
ax.plot(result.times, result.theoretical_max_powers, '--', label='Theoretical MPP', linewidth=2)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Power (W)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/figures/notebook_reproduced_trajectory.png', dpi=300)
plt.show()

In [ ]:
efficiency = 100.0 * np.trapz(result.powers, result.times) / np.trapz(result.theoretical_max_powers, result.times)
print(f"Calculated tracking efficiency: {efficiency:.2f}%")
print("Compare against results/summary_table.csv (algorithm=p_and_o, scenario=steady_state, metric=tracking_efficiency_pct)")